In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install -U diffusers transformers accelerate peft safetensors xformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 44.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 89.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 66.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 17.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 77.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface

In [3]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.9 MB/s eta 0:00:00:00:0100:01


# Data Loader

In [7]:
import os
from PIL import Image
from torch.utils.data import Dataset

In [6]:
class Img2ImgDataset(Dataset):
    def __init__(self, input_dir, target_dir, size=512):
        self.input_dir = input_dir
        self.target_dir = target_dir
        self.size = size
        self.files = sorted(os.listdir(input_dir))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]

        inp = Image.open(os.path.join(self.input_dir, fname)).convert("RGB")
        tgt = Image.open(os.path.join(self.target_dir, fname)).convert("RGB")

        inp = inp.resize((self.size, self.size))
        tgt = tgt.resize((self.size, self.size))

        return {
            "input_image": inp,
            "target_image": tgt
        }

/kaggle/working


# Load S.D with LoRA setup

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

In [ ]:
model_name = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_name,
    torch_dtype=torch.float16
).to("cuda")

## Enable LoRA layers

In [ ]:
from peft import LoraConfig, get_peft_model

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
    lora_dropout=0.1,
    bias="none"
)

pipe.unet = get_peft_model(pipe.unet, lora_config)

## Training

In [8]:
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torchvision import transforms

In [ ]:
dataset = Img2ImgDataset(
    "/kaggle/input/dataset/input",
    "/kaggle/input/dataset/target"
)

loader = DataLoader(dataset, batch_size=1, shuffle=True)

optimizer = torch.optim.AdamW(pipe.unet.parameters(), lr=1e-4)

transform = transforms.ToTensor()

pipe.unet.train()

In [ ]:
for epoch in range(3):
    for batch in loader:

        input_img = transform(batch["input_image"][0]).unsqueeze(0).cuda()
        target_img = transform(batch["target_image"][0]).unsqueeze(0).cuda()

        noise = torch.randn_like(target_img)
        timesteps = torch.randint(0, 1000, (1,), device="cuda")

        noisy = target_img + noise * 0.1

        pred = pipe.unet(noisy, timesteps).sample

        loss = F.mse_loss(pred, target_img)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    print(f"Epoch {epoch} loss: {loss.item()}")

## save model